In [2]:
import numpy as np
import pandas as pd

# new_cell_0.py
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
)

# Assumes y_test, xgb_probs_test, ae_errors_test are defined in notebook (cells below can be run before this one).
# Try to safely fetch variables from the notebook/user namespace and handle missing data without raising NameError.
from IPython import get_ipython

def _get_nb_var(name):
    try:
        ip = get_ipython()
        if ip is not None and name in ip.user_ns:
            return ip.user_ns[name]
    except Exception:
        pass
    return globals().get(name, None)

y_test = _get_nb_var("y_test")
xgb_probs_test = _get_nb_var("xgb_probs_test")
ae_errors_test = _get_nb_var("ae_errors_test")

if y_test is None or xgb_probs_test is None or ae_errors_test is None:
    missing = [n for n, v in [("y_test", y_test), ("xgb_probs_test", xgb_probs_test), ("ae_errors_test", ae_errors_test)] if v is None]
    print(f"Missing variables in notebook: {missing}. Please run the cells that define them before this evaluation cell.")
    # Use empty placeholders to avoid NameError; downstream logic will skip computations.
    y_true = np.array([], dtype=int)
    xgb_prob = np.array([], dtype=float)
    ae_err = np.array([], dtype=float)
    data_available = False
else:
    # Convert to numpy arrays for safety
    y_true = np.asarray(y_test).astype(int)
    xgb_prob = np.asarray(xgb_probs_test).astype(float)
    ae_err = np.asarray(ae_errors_test).astype(float)
    data_available = True

# If there's no data, skip metric computation and print a friendly message (no traceback).
if not data_available or y_true.size == 0:
    print("No data available; skipping metric computation.")
    print("\nComparison table:")
    print(pd.DataFrame(columns=["precision", "recall", "f1", "roc_auc", "average_precision"]))
    metrics = {"xgb": {}, "ae": {}, "hybrid": {}}
    preds = {"y_pred_xgb": np.array([], dtype=int), "y_pred_ae": np.array([], dtype=int), "y_pred_hybrid": np.array([], dtype=int)}
    scores = {"xgb_prob": np.array([], dtype=float), "ae_err": np.array([], dtype=float), "hybrid_score": np.array([], dtype=float)}
else:
    # 1) Build hybrid predictions according to provided logic
    y_pred_hybrid = np.zeros_like(y_true, dtype=int)

    # Gray zone indices
    mask_approve = xgb_prob < 0.05
    mask_block = xgb_prob >= 0.80
    mask_gray = ~(mask_approve | mask_block)

    y_pred_hybrid[mask_approve] = 0
    y_pred_hybrid[mask_block] = 1

    # Apply AE thresholds inside gray zone:
    # - ae_error >= 4.896 -> 1
    # - else if ae_error >= 0.692 -> 1 (review treated as fraud for evaluation)
    # - else -> 0
    ae_high_mask = (ae_err >= 4.896) & mask_gray
    ae_review_mask = (ae_err >= 0.692) & mask_gray & ~ae_high_mask
    y_pred_hybrid[ae_high_mask] = 1
    y_pred_hybrid[ae_review_mask] = 1
    # remaining gray zone entries already 0

    # 2) Compute metrics helper
    def compute_metrics(y_true, y_pred, score=None):
        out = {}
        out["precision"] = precision_score(y_true, y_pred, zero_division=0)
        out["recall"] = recall_score(y_true, y_pred, zero_division=0)
        out["f1"] = f1_score(y_true, y_pred, zero_division=0)
        # ROC-AUC & Average Precision require continuous scores and at least two classes in y_true
        if score is not None and len(np.unique(y_true)) > 1:
            try:
                out["roc_auc"] = roc_auc_score(y_true, score)
            except Exception:
                out["roc_auc"] = np.nan
            try:
                out["average_precision"] = average_precision_score(y_true, score)
            except Exception:
                out["average_precision"] = np.nan
        else:
            out["roc_auc"] = np.nan
            out["average_precision"] = np.nan
        return out

    # XGB: predictions using 0.5 threshold for discrete metrics; use xgb_prob as score for AUC/AP
    y_pred_xgb = (xgb_prob >= 0.5).astype(int)
    metrics_xgb = compute_metrics(y_true, y_pred_xgb, score=xgb_prob)

    # AE: treat review as fraud for evaluation: threshold >= 0.692 -> fraud
    y_pred_ae = (ae_err >= 0.692).astype(int)
    # Use raw ae_err as score for AUC/AP (higher error -> more likely fraud)
    metrics_ae = compute_metrics(y_true, y_pred_ae, score=ae_err)

    # Hybrid: discrete predictions already computed. Build a proxy continuous score for ROC/AUPR:
    # - decisive XGB cases map to 0 or 1
    # - gray zone: map to normalized AE error between 0 and 1
    ae_min, ae_max = ae_err.min(), ae_err.max()
    if ae_max > ae_min:
        ae_scaled = (ae_err - ae_min) / (ae_max - ae_min)
    else:
        ae_scaled = np.zeros_like(ae_err, dtype=float)

    hybrid_score = np.where(mask_approve, 0.0, np.where(mask_block, 1.0, ae_scaled))
    metrics_hybrid = compute_metrics(y_true, y_pred_hybrid, score=hybrid_score)

    # 3) Print classification report for hybrid
    print("Hybrid classification report (review treated as fraud):")
    print(classification_report(y_true, y_pred_hybrid, digits=4, zero_division=0))

    # 4) Comparison table
    results = pd.DataFrame(
        {
            "XGB": metrics_xgb,
            "AE": metrics_ae,
            "Hybrid": metrics_hybrid,
        }
    ).T[["precision", "recall", "f1", "roc_auc", "average_precision"]]

    # Format numeric values
    results = results.applymap(lambda v: float(f"{v:.4f}") if (pd.notnull(v) and not np.isnan(v)) else np.nan)

    print("\nComparison table:")
    print(results)

    # Optionally return objects for further inspection
    # (In a notebook you can capture these variables)
    metrics = {"xgb": metrics_xgb, "ae": metrics_ae, "hybrid": metrics_hybrid}
    preds = {"y_pred_xgb": y_pred_xgb, "y_pred_ae": y_pred_ae, "y_pred_hybrid": y_pred_hybrid}
    scores = {"xgb_prob": xgb_prob, "ae_err": ae_err, "hybrid_score": hybrid_score}

Missing variables in notebook: ['y_test', 'xgb_probs_test', 'ae_errors_test']. Please run the cells that define them before this evaluation cell.
No data available; skipping metric computation.

Comparison table:
Empty DataFrame
Columns: [precision, recall, f1, roc_auc, average_precision]
Index: []
